# Frozen G3C R4 official 1,012 — reranker shards 2 + 3

Attach exactly the frozen official payload and the validated embedding-result dataset. This notebook is the second independent reranker phase and never reads labels or public scores.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
payload_matches = sorted(Path('/kaggle/input').rglob('g3c_official_payload_manifest.json'))
embedding_matches = sorted(Path('/kaggle/input').rglob('g3c_official_embedding_manifest.json'))
assert len(payload_matches) == 1 and len(embedding_matches) == 1, (payload_matches, embedding_matches)
PAYLOAD = payload_matches[0].parent
EMBEDDING = embedding_matches[0].parent
manifest = json.loads(payload_matches[0].read_text(encoding='utf-8'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PAYLOAD / manifest['paths']['requirements'])], check=True)
print({'payload': str(PAYLOAD), 'embedding': str(EMBEDDING), 'shards': [2, 3]})

In [ ]:
os.environ['HF_HOME'] = '/kaggle/temp/g3c_official_hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
sys.dont_write_bytecode = True
sys.path.insert(0, str(PAYLOAD / 'code'))
import torch, transformers
from vifinqa.g3c_official.payload import validate_official_payload
from vifinqa.g3c_official.execution import validate_embedding_result
assert transformers.__version__ == '4.53.3'
assert torch.cuda.is_available() and torch.cuda.device_count() == 2
gpu = [(torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i), torch.cuda.get_device_properties(i).total_memory) for i in range(2)]
assert gpu[0] == gpu[1], gpu
validate_official_payload(PAYLOAD)
validate_embedding_result(payload_dir=PAYLOAD, result_dir=EMBEDDING, require_qwen=True)
print({'gpus': gpu, 'inputs_valid': True})

In [ ]:
OUT = Path('/kaggle/working/g3c_official_rerank_b')
runner = PAYLOAD / manifest['paths']['runner']
command = [sys.executable, str(runner), 'rerank-pair', '--payload', str(PAYLOAD), '--embedding-results', str(EMBEDDING), '--out', str(OUT), '--shards', '2', '3', '--backend', 'qwen']
print('Starting exact two-T4 reranker shards 2 + 3')
subprocess.run(command, check=True)

In [ ]:
from vifinqa.g3c_official.execution import validate_rerank_pair_result
report = validate_rerank_pair_result(payload_dir=PAYLOAD, embedding_result_dir=EMBEDDING, result_dir=OUT, expected_shards=(2, 3), require_qwen=True)
assert report['exact_canary_passed_on_both_gpus'] is True
print(json.dumps({'run_signature': report['run_signature'], 'shards': report['shard_indices'], 'seconds': report['total_seconds']}, indent=2))

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/g3c_official_rerank_b_results', 'zip', root_dir=OUT)
print({'download_directory': str(OUT), 'download_zip': archive})